In [15]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/movie-recommendation-hackathon-2026/sample_submission.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/movies.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/imdb_data.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/genome_tags.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/genome_scores.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/train.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/test.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/tags.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/links.csv


In [16]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split

In [17]:
train_df = pd.read_csv('/kaggle/input/competitions/movie-recommendation-hackathon-2026/train.csv')

train_df.head()

,userId,movieId,rating,timestamp
0,5163,57669,4.0,1518349992
1,106343,5,4.5,1206238739
2,146790,5459,5.0,1076215539
3,106362,32296,2.0,1423042565
4,9041,366,3.0,833375837


In [18]:
tags_df = pd.read_csv('/kaggle/input/competitions/movie-recommendation-hackathon-2026/tags.csv')

tags_df.head()

,userId,movieId,tag,timestamp
0,3,260,classic,1439472355
1,3,260,sci-fi,1439472256
2,4,1732,dark comedy,1573943598
3,4,1732,great dialogue,1573943604
4,4,7569,so bad it's good,1573943455


In [19]:
movies_df = pd.read_csv('/kaggle/input/competitions/movie-recommendation-hackathon-2026/movies.csv')

movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [20]:
links_df = pd.read_csv('/kaggle/input/competitions/movie-recommendation-hackathon-2026/links.csv')

links_df.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [21]:
genome_scores_df = pd.read_csv('/kaggle/input/competitions/movie-recommendation-hackathon-2026/genome_scores.csv')

genome_scores_df.head()

,movieId,tagId,relevance
0,1,1,0.02875
1,1,2,0.02375
2,1,3,0.06250
3,1,4,0.07575
4,1,5,0.14075


In [22]:
genome_tags_df = pd.read_csv('/kaggle/input/competitions/movie-recommendation-hackathon-2026/genome_tags.csv')

genome_tags_df.head()

,tagId,tag
0,1,007
1,2,007 (series)
2,3,18th century
3,4,1920s
4,5,1930s


In [23]:
sample_submission_df = pd.read_csv('/kaggle/input/competitions/movie-recommendation-hackathon-2026/sample_submission.csv')

sample_submission_df.head()

,Id,rating
0,1_2011,1.0
1,1_4144,1.0
2,1_5767,1.0
3,1_6711,1.0
4,1_7318,1.0


## Collaborative filtering on users using cosine similarity

In [24]:
# user_item_matrix = train_df.pivot_table(
#     index='userId',
#     columns='movieId',
#     values='rating'
# )

user_codes, user_ids = pd.factorize(train_df['userId'], sort=True)
movie_codes, movie_ids = pd.factorize(train_df['movieId'], sort=True)

user_id_to_idx = pd.Series(
    np.arange(len(user_ids)),
    index=user_ids
)

movie_id_to_idx = pd.Series(
    np.arange(len(movie_ids)),
    index=movie_ids
)

user_item_matrix = csr_matrix(
    (
        train_df['rating'].to_numpy(dtype=np.float32),
        (user_codes, movie_codes)
    ),
    shape=(len(user_ids), len(movie_ids))
)

In [25]:
def get_user_similarities(user_item_matrix, user_id, user_id_to_idx, user_ids, movie_id_to_idx, movie_ids, k=20):
    # Check whether the user exists
    if user_id not in user_id_to_idx.index:
        raise ValueError(f"User {user_id} not found in training data")

    # Get the target user's matrix position
    user_idx = user_id_to_idx[user_id]

    # Extract the target user's rating vector
    user_vector = user_item_matrix[user_idx]

    # Calculate similarity against all users
    similarities = cosine_similarity(
        user_vector,
        user_item_matrix
    ).flatten()

    # Convert similarities into a Series indexed by original user IDs
    user_similarities = pd.Series(
        similarities,
        index=user_ids
    )

    # Exclude the target user and retrieve the top-K users
    top_k_users = (
        user_similarities.drop(user_id).sort_values(ascending=False).head(k)
    )

    # Retrieve movies already watched by the target user
    watched_movies = movie_ids[user_vector.indices]

    # Retrieve the matrix positions of the top-K users
    similar_user_indices = user_id_to_idx.loc[
        top_k_users.index
    ].to_numpy()

    # Extract the top-K users' rating vectors
    similar_users_matrix = user_item_matrix[similar_user_indices]

    # Identify movies rated by the top-K users
    candidate_positions = np.unique(
        similar_users_matrix.indices
    )

    candidate_movies = movie_ids[candidate_positions]

    # Exclude movies already watched by the target user
    unwatched_candidates = np.setdiff1d(
        candidate_movies,
        watched_movies
    )

    candidate_movie_indices = movie_id_to_idx.loc[
        unwatched_candidates
    ].to_numpy()

    neighbor_ratings = similar_users_matrix[:, candidate_movie_indices]
    similarity_weights = top_k_users.to_numpy().reshape(-1, 1)

    weighted_rating_sums = np.asarray(
        neighbor_ratings.T @ similarity_weights
    ).ravel()

    rated_mask = (neighbor_ratings != 0).astype(np.float32)
    similarity_sums = np.asarray(
        rated_mask.T @ similarity_weights
    ).ravel()

    predicted_ratings = np.divide(
        weighted_rating_sums,
        similarity_sums,
        out=np.full(len(unwatched_candidates), np.nan),
        where=similarity_sums != 0
    )

    recommendations = pd.DataFrame({
        'movieId': unwatched_candidates,
        'predicted_rating': predicted_ratings
    })

    recommendations = recommendations.sort_values(
        by='predicted_rating',
        ascending=False
    )

    recommendations = recommendations.merge(
        movies_df[['movieId', 'title']],
        on='movieId',
        how='left'
    )

    return top_k_users, recommendations
    # user_similarities = user_similarities.drop(user_id)

    # return user_similarities.sort_values(ascending=False)

    # # Calculate similarity between the target user and all other users
    # user_vector = user_item_matrix.loc[[user_id]]

    # similarities = cosine_similarity(
    #     user_vector,
    #     user_item_matrix
    # )

    # user_similarities = pd.Series(
    #     similairities.flatten(),
    #     index=user_item_matrix.index
    # )

    # # Exclude the target user
    # user_similarities = user_similairities.drop(user_id)
    
    # return user_similarities.sort_values(ascending=False).head(k)

# user_item_matrix.head()

In [26]:
top_k_users, recommendations = get_user_similarities(
    user_item_matrix=user_item_matrix,
    user_id=101,
    user_id_to_idx=user_id_to_idx,
    user_ids=user_ids,
    movie_ids=movie_ids,
    movie_id_to_idx=movie_id_to_idx,k=20
)

In [27]:
top_k_users

34229     0.362976
24999     0.361584
14217     0.337938
123127    0.323275
124887    0.320462
161036    0.319364
56560     0.311886
32004     0.311286
148241    0.309004
86699     0.308668
151516    0.308210
32784     0.308082
77470     0.305766
96637     0.304746
110415    0.303323
82194     0.302300
71494     0.301820
111968    0.299218
7425      0.298699
57329     0.297506
dtype: float32

In [29]:
user_similarities = get_user_similarities(
    user_item_matrix,
    user_id=101,
    user_id_to_idx=user_id_to_idx,
    user_ids=user_ids
)

user_similarities.head(10)

TypeError: get_user_similarities() missing 2 required positional arguments: 'movie_id_to_idx' and 'movie_ids'

## Evaluating the algorithm

In [30]:
train_data, validation_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42
)

user_codes, user_ids = pd.factorize(train_data['userId'], sort=True)
movie_codes, movie_ids = pd.factorize(train_data['movieId'], sort=True)

user_id_to_idx = pd.Series(
    np.arange(len(user_ids)),
    index=user_ids
)

movie_id_to_idx = pd.Series(
    np.arange(len(movie_ids)),
    index=movie_ids
)

user_item_matrix = csr_matrix(
    (
        train_data['rating'].to_numpy(dtype=np.float32),
        (user_codes, movie_codes)
    ),
    shape=(len(user_ids), len(movie_ids))
)

In [31]:
def get_user_similarities(
    user_item_matrix,
    user_id,
    user_id_to_idx,
    user_ids,
    movie_ids,
    movie_id_to_idx,
    k=20,
    movies_to_predict=None
):
    # Check whether the user exists
    if user_id not in user_id_to_idx.index:
        raise ValueError(f"User {user_id} not found in training data")

    # Get the target user's matrix position
    user_idx = user_id_to_idx[user_id]

    # Extract the target user's rating vector
    user_vector = user_item_matrix[user_idx]

    # Calculate similarity against all users
    similarities = cosine_similarity(
        user_vector,
        user_item_matrix
    ).flatten()

    # Convert similarities into a Series indexed by original user IDs
    user_similarities = pd.Series(
        similarities,
        index=user_ids
    )

    # Exclude the target user and retrieve the top-K users
    top_k_users = (
        user_similarities.drop(user_id).sort_values(ascending=False).head(k)
    )

    # Retrieve movies already watched by the target user
    watched_movies = movie_ids[user_vector.indices]

    # Retrieve the matrix positions of the top-K users
    similar_user_indices = user_id_to_idx.loc[
        top_k_users.index
    ].to_numpy()

    # Extract the top-K users' rating vectors
    similar_users_matrix = user_item_matrix[similar_user_indices]

    if movies_to_predict is None:
        # Identify movies rated by the top-K users
        candidate_positions = np.unique(
            similar_users_matrix.indices
        )

        candidate_movies = movie_ids[candidate_positions]

        # Exclude movies already watched by the target user
        unwatched_candidates = np.setdiff1d(
            candidate_movies,
            watched_movies
        )

    else:
        unwatched_candidates = np.asarray(movies_to_predict)

    candidate_movie_indices = movie_id_to_idx.loc[
        unwatched_candidates
    ].to_numpy()

    neighbor_ratings = similar_users_matrix[:, candidate_movie_indices]
    similarity_weights = top_k_users.to_numpy().reshape(-1, 1)

    weighted_rating_sums = np.asarray(
        neighbor_ratings.T @ similarity_weights
    ).ravel()

    rated_mask = (neighbor_ratings != 0).astype(np.float32)
    similarity_sums = np.asarray(
        rated_mask.T @ similarity_weights
    ).ravel()

    predicted_ratings = np.divide(
        weighted_rating_sums,
        similarity_sums,
        out=np.full(len(unwatched_candidates), np.nan),
        where=similarity_sums != 0
    )

    recommendations = pd.DataFrame({
        'movieId': unwatched_candidates,
        'predicted_rating': predicted_ratings
    })

    recommendations = recommendations.sort_values(
        by='predicted_rating',
        ascending=False
    )

    recommendations = recommendations.merge(
        movies_df[['movieId', 'title']],
        on='movieId',
        how='left'
    )

    return top_k_users, recommendations

In [32]:
validation_eval_data = validation_data[
    validation_data['userId'].isin(user_id_to_idx.index) &
    validation_data['movieId'].isin(movie_id_to_idx.index)
].copy()

In [ ]:
# Store predictions from all validation users
all_predictions = []

for user_id, user_validation in validation_eval_data.groupby('userId'):
    # Predict ratings for this user's validation movies
    _, user_predictions = get_user_similarities(
        user_item_matrix=user_item_matrix,
        user_id=user_id,
        user_id_to_idx=user_id_to_idx,
        user_ids=user_ids,
        movie_ids=movie_ids,
        movie_id_to_idx=movie_id_to_idx,
        k=20,
        movies_to_predict=user_validation['movieId'].to_numpy()
    )

    # Attach the corresponding user ID
    user_predictions['userId'] = user_id
    all_predictions.append(user_predictions)

# Combine predictions from all users
predictions_df = pd.concat(
    all_predictions,
    ignore_index=True
)

In [ ]:
evaluation_df = validation_eval_data.merge(
    predictions_df[['userId', 'movieId', 'predicted_rating']],
    on=['userId', 'movieId'],
    how='inner'
)

In [ ]:
evaluation_df = evaluation_df.dropna(
    subset=['predicted_rating']
)

rmse = root_mean_squared_error(
    evaluation_df['rating'],
    evaluation_df['predicted_rating']
)

print("RMSE: ", rmse)

covarage = len(evaluation_df) / len(validation_eval_data)
print("Prediction coverage: ", coverage * 100, "%")